# PySpark

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-interview-questions/blob/master/notebooks/topics/spark.ipynb)

*How to use this notebook.* Read the explanation cells and run the code cells below them. Python examples list their imports, so install what a section needs before running it. SQL and shell snippets are illustrative: they show the syntax for the platform named in that section and need that platform (or a compatible engine) to run.

## What is PySpark and where does it fit in a data platform?

PySpark is the Python API for Apache Spark. It lets you write distributed data processing jobs in Python while Spark executes the work across a cluster.

In a modern lakehouse, PySpark is often used for:

+ Reading raw files from S3, ADLS, GCS, Kafka, or relational sources
+ Cleaning and standardizing semi-structured data
+ Joining large fact data with dimensions
+ Writing Delta, Iceberg, or Parquet tables for analytics
+ Running batch pipelines, streaming pipelines, and feature engineering jobs

Typical production flow:

```text
raw source files -> PySpark bronze job -> PySpark silver job -> Unity Catalog gold tables -> BI / ML / reverse ETL
```

Real-world example: an orders pipeline lands JSON in S3 every 15 minutes. A PySpark job reads the raw JSON, enforces a schema, deduplicates by `order_id`, joins customer data, and writes a governed table such as `main.gold.fact_order`.

## How do you create a SparkSession in PySpark?

`SparkSession` is the main entry point for DataFrame, SQL, catalog, and streaming operations. In Databricks notebooks, `spark` is usually already available. In jobs, tests, or local development, you create it explicitly.

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("orders-pipeline")
    .config("spark.sql.shuffle.partitions", "200")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

In Databricks with Unity Catalog, you normally refer to tables with a three-part name:

In [ ]:
orders = spark.table("main.silver.orders")

Interview point: `SparkSession` replaced older entry points like `SQLContext` for most DataFrame and SQL work. `SparkContext` still exists underneath, but most production PySpark code should use `SparkSession` and DataFrames.

## How do you read data with explicit schemas in PySpark?

Explicit schemas make pipelines safer and faster. If you let Spark infer schemas from files, the schema may change based on a small sample, a malformed record, or a new partition.

In [ ]:
from pyspark.sql import types as T

order_schema = T.StructType([
    T.StructField("order_id", T.StringType(), False),
    T.StructField("customer_id", T.StringType(), True),
    T.StructField("order_ts", T.TimestampType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("amount", T.DecimalType(12, 2), True),
])

orders_raw = (
    spark.read
    .schema(order_schema)
    .option("mode", "PERMISSIVE")
    .json("s3://company-raw/orders/ds=2026-10-04/")
)

For CSV, be explicit about headers, delimiters, quotes, and bad-record handling:

In [ ]:
customers = (
    spark.read
    .schema("customer_id string, email string, created_at timestamp")
    .option("header", "true")
    .option("escape", '"')
    .csv("s3://company-raw/customers/")
)

Real-world example: if `amount` is accidentally delivered as `"N/A"` in one file, schema inference can degrade the column to string. With an explicit decimal schema, invalid rows become visible as quality failures instead of silently changing the table contract.

## How do DataFrames differ from RDDs in PySpark?

DataFrames are structured, schema-aware distributed datasets. Spark can optimize DataFrame operations using Catalyst and Tungsten. RDDs are lower-level distributed collections where Spark sees less structure and has fewer optimization opportunities.

Prefer DataFrames for most data engineering work:

In [ ]:
from pyspark.sql import functions as F

paid_orders = (
    spark.table("main.silver.orders")
    .where(F.col("status") == "paid")
    .select("order_id", "customer_id", "amount", "order_ts")
)

Use RDDs only when you truly need record-level custom distributed logic that DataFrame APIs cannot express:

In [ ]:
line_lengths = spark.sparkContext.textFile("s3://company-raw/logs/").map(len)

Interview answer: DataFrames are usually better for production pipelines because they support SQL optimization, column pruning, predicate pushdown, better execution planning, and integration with table formats.

## What are transformations and actions in PySpark?

Transformations build a logical plan. Actions trigger execution.

Transformations:

In [ ]:
from pyspark.sql import functions as F

daily_revenue = (
    spark.table("main.silver.orders")
    .where(F.col("status").isin("paid", "shipped"))
    .withColumn("order_date", F.to_date("order_ts"))
    .groupBy("order_date")
    .agg(F.sum("amount").alias("revenue"))
)

No cluster job has to run yet. An action triggers execution:

In [ ]:
daily_revenue.count()
daily_revenue.write.mode("overwrite").saveAsTable("main.gold.daily_revenue")

Common transformations include `select`, `where`, `withColumn`, `join`, `groupBy`, `agg`, `distinct`, and `dropDuplicates`. Common actions include `count`, `collect`, `show`, `take`, `write`, and `foreachBatch`.

## How does lazy evaluation work in PySpark?

Lazy evaluation means Spark waits until an action before executing the plan. This lets Spark optimize the whole plan instead of executing every line immediately.

In [ ]:
orders = spark.table("main.silver.orders")

result = (
    orders
    .where("order_ts >= timestamp '2026-10-01'")
    .select("order_id", "customer_id", "amount")
    .where("amount > 0")
)

result.explain("formatted")

Spark can combine filters, prune unused columns, push predicates to Parquet/Delta/Iceberg readers, and choose join strategies before running tasks.

Real-world example: if you read a 2 TB orders table but select only `order_id` and `amount`, Spark can avoid reading unused columns from Parquet files. That optimization is possible because the plan is declarative.

## How do you use Spark SQL and temp views from PySpark?

PySpark and Spark SQL work together. DataFrames can be registered as temporary views, and SQL results come back as DataFrames.

In [ ]:
orders = spark.table("main.silver.orders")
orders.createOrReplaceTempView("orders")

daily = spark.sql("""
select
  date(order_ts) as order_date,
  count(*) as order_count,
  sum(amount) as revenue
from orders
where status in ('paid', 'shipped')
group by date(order_ts)
""")

For governed production tables, use Unity Catalog names directly:

In [ ]:
spark.sql("""
create or replace table main.gold.daily_revenue as
select
  date(order_ts) as order_date,
  sum(amount) as revenue
from main.silver.orders
where status in ('paid', 'shipped')
group by date(order_ts)
""")

Interview point: PySpark DataFrame code and Spark SQL usually compile into the same execution engine. Choose the style that is clearer for the team and workload.

## How do you clean and standardize data in PySpark?

Use built-in column functions instead of Python row loops. Built-ins run inside Spark's optimized execution engine.

In [ ]:
from pyspark.sql import functions as F

clean_customers = (
    spark.table("main.bronze.customers")
    .select(
        F.trim(F.lower("email")).alias("email"),
        F.upper(F.trim("country_code")).alias("country_code"),
        F.to_timestamp("created_at").alias("created_at"),
        F.regexp_replace("phone", r"[^0-9]", "").alias("phone_digits"),
    )
    .where(F.col("email").contains("@"))
)

For reusable business logic, wrap DataFrame transformations in plain Python functions:

In [ ]:
def standardize_orders(df):
    return (
        df
        .withColumn("status", F.lower(F.trim("status")))
        .withColumn("order_date", F.to_date("order_ts"))
        .where(F.col("order_id").isNotNull())
    )

orders = standardize_orders(spark.table("main.bronze.orders"))

This style is testable, readable, and avoids hiding Spark logic inside side-effect-heavy notebooks.

## What are common PySpark data manipulation examples?

Most PySpark pipeline work is ordinary data manipulation: parsing timestamps, deriving dates, handling nulls, standardizing strings, splitting arrays, and creating business flags. Prefer `pyspark.sql.functions` because those expressions stay inside Spark's optimizer.

Start with a realistic orders DataFrame:

In [ ]:
from pyspark.sql import functions as F

orders = spark.table("main.bronze.orders")

Date and timestamp manipulation:

In [ ]:
orders_with_dates = (
    orders
    .withColumn("order_ts", F.to_timestamp("order_ts"))
    .withColumn("order_date", F.to_date("order_ts"))
    .withColumn("order_month", F.date_trunc("month", "order_ts"))
    .withColumn("order_year", F.year("order_ts"))
    .withColumn("order_week", F.weekofyear("order_ts"))
    .withColumn("ship_by_date", F.date_add("order_date", 2))
    .withColumn("days_since_order", F.datediff(F.current_date(), "order_date"))
)

Timezone conversion is common when sources emit UTC but reporting is local:

In [ ]:
localized = (
    orders
    .withColumn("order_ts_utc", F.to_timestamp("order_ts"))
    .withColumn(
        "order_ts_chicago",
        F.from_utc_timestamp("order_ts_utc", "America/Chicago"),
    )
    .withColumn("order_date_chicago", F.to_date("order_ts_chicago"))
)

String cleanup:

In [ ]:
clean_customers = (
    spark.table("main.bronze.customers")
    .withColumn("email", F.lower(F.trim("email")))
    .withColumn("first_name", F.initcap(F.trim("first_name")))
    .withColumn("phone_digits", F.regexp_replace("phone", r"[^0-9]", ""))
    .withColumn("email_domain", F.element_at(F.split("email", "@"), -1))
)

Null handling and default values:

In [ ]:
orders_filled = (
    orders
    .withColumn("discount_amount", F.coalesce("discount_amount", F.lit(0.0)))
    .withColumn("status", F.coalesce("status", F.lit("unknown")))
    .fillna({"country_code": "US", "currency": "USD"})
)

Type casting and safe derived columns:

In [ ]:
typed_orders = (
    orders
    .withColumn("amount", F.col("amount").cast("decimal(12,2)"))
    .withColumn("quantity", F.col("quantity").cast("int"))
    .withColumn(
        "unit_price",
        F.when(F.col("quantity") > 0, F.col("amount") / F.col("quantity")),
    )
)

Conditional business flags:

In [ ]:
classified = (
    orders
    .withColumn(
        "order_size",
        F.when(F.col("amount") >= 1000, "enterprise")
         .when(F.col("amount") >= 100, "mid_market")
         .otherwise("small"),
    )
    .withColumn(
        "is_completed",
        F.col("status").isin("paid", "shipped", "delivered"),
    )
)

Array and nested field manipulation:

In [ ]:
events = spark.table("main.bronze.events")

items = (
    events
    .select(
        "event_id",
        F.col("user.id").alias("user_id"),
        F.explode_outer("items").alias("item"),
    )
    .select(
        "event_id",
        "user_id",
        F.col("item.sku").alias("sku"),
        F.col("item.quantity").alias("quantity"),
    )
)

Parsing JSON strings:

In [ ]:
payload_schema = "source string, campaign struct<id:string,name:string>"

parsed = (
    events
    .withColumn("payload", F.from_json("payload_json", payload_schema))
    .select(
        "event_id",
        F.col("payload.source").alias("source"),
        F.col("payload.campaign.id").alias("campaign_id"),
    )
)

Deduplicating records:

In [ ]:
deduped_orders = (
    orders
    .dropDuplicates(["order_id"])
)

For CDC or late-arriving updates, use a window so you keep the newest record by source ordering:

In [ ]:
from pyspark.sql import Window

w = Window.partitionBy("order_id").orderBy(
    F.col("source_lsn").desc(),
    F.col("ingested_at").desc(),
)

latest_orders = (
    orders
    .withColumn("rn", F.row_number().over(w))
    .where("rn = 1")
    .drop("rn")
)

Real-world guidance: if the transformation is expressible with built-in functions, use built-in functions. They are easier for Spark to optimize than Python loops or UDFs.

## How do you join data in PySpark?

A join combines DataFrames by key. The most important production questions are key uniqueness, null handling, data skew, and whether Spark should shuffle both sides or broadcast the small side.

In [ ]:
orders = spark.table("main.silver.orders")
customers = spark.table("main.silver.customers")

enriched = (
    orders.alias("o")
    .join(customers.alias("c"), on="customer_id", how="left")
    .select(
        "o.order_id",
        "o.customer_id",
        "c.customer_segment",
        "o.order_ts",
        "o.amount",
    )
)

If the dimension is small enough, broadcast it:

In [ ]:
from pyspark.sql.functions import broadcast

enriched = orders.join(broadcast(customers), "customer_id", "left")

Real-world example: broadcasting a 20 MB customer segment dimension can avoid a large shuffle when joining it to a 3 TB orders fact table. Do not broadcast a dimension just because it is called a dimension; verify its size.

## How do you aggregate data in PySpark?

Aggregations summarize records by one or more grouping columns. They usually create a shuffle because rows with the same key must be brought together.

In [ ]:
from pyspark.sql import functions as F

daily_product_sales = (
    spark.table("main.gold.fact_order_line")
    .groupBy("order_date", "product_id")
    .agg(
        F.countDistinct("order_id").alias("orders"),
        F.sum("quantity").alias("units"),
        F.sum("net_amount").alias("revenue"),
    )
)

For conditional metrics, use expressions:

In [ ]:
customer_metrics = (
    spark.table("main.gold.fact_order")
    .groupBy("customer_id")
    .agg(
        F.sum("amount").alias("lifetime_value"),
        F.sum(F.when(F.col("status") == "refunded", F.col("amount")).otherwise(0)).alias("refunded_amount"),
        F.max("order_ts").alias("last_order_ts"),
    )
)

Interview point: aggregations often need tuning because they shuffle data. Check the Spark UI for skewed shuffle partitions, spilled memory, and long-running reduce tasks.

## How do window functions work in PySpark?

Window functions calculate values across related rows without collapsing the result like `groupBy` does.

In [ ]:
from pyspark.sql import Window, functions as F

w = Window.partitionBy("customer_id").orderBy(F.col("order_ts").desc())

latest_order = (
    spark.table("main.silver.orders")
    .withColumn("rn", F.row_number().over(w))
    .where(F.col("rn") == 1)
    .drop("rn")
)

Common window use cases:

+ Deduplicating CDC records by latest sequence number
+ Finding the latest row per business key
+ Calculating rolling metrics
+ Ranking products or customers within a segment

CDC example:

In [ ]:
w = Window.partitionBy("order_id").orderBy(
    F.col("source_lsn").desc(),
    F.col("ingested_at").desc(),
)

deduped_cdc = (
    spark.table("main.bronze.orders_cdc")
    .withColumn("rn", F.row_number().over(w))
    .where("rn = 1")
    .drop("rn")
)

## How do you handle nested JSON and arrays in PySpark?

Nested data is common in event, clickstream, API, and CDC payloads. Use struct and array functions instead of converting rows to Python dictionaries.

In [ ]:
from pyspark.sql import functions as F

events = spark.read.json("s3://company-raw/events/")

flattened = events.select(
    F.col("event_id"),
    F.col("user.id").alias("user_id"),
    F.col("device.os").alias("device_os"),
    F.explode_outer("items").alias("item"),
)

items = flattened.select(
    "event_id",
    "user_id",
    F.col("item.sku").alias("sku"),
    F.col("item.quantity").alias("quantity"),
)

If a payload column contains JSON strings, parse it with `from_json`:

In [ ]:
payload_schema = "event_type string, campaign struct<id:string,name:string>"

parsed = (
    spark.table("main.bronze.raw_events")
    .withColumn("payload", F.from_json("payload_json", payload_schema))
    .select("event_id", "payload.event_type", "payload.campaign.id")
)

Interview point: `explode` can multiply rows. Always know the expected array size and watch for unexpected row explosions.

## How do you write PySpark tables in Databricks Unity Catalog?

With Unity Catalog, prefer managed table names like `catalog.schema.table` over raw paths in application code. This keeps governance, lineage, discovery, and permissions at the table layer.

In [ ]:
spark.sql("create schema if not exists main.gold")

(
    daily_product_sales
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("main.gold.daily_product_sales")
)

For partitioned Delta tables:

In [ ]:
(
    daily_product_sales
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("order_date")
    .saveAsTable("main.gold.daily_product_sales")
)

For table-format-aware writes, use `writeTo` when the target table already exists:

In [ ]:
(
    daily_product_sales
    .writeTo("main.gold.daily_product_sales")
    .append()
)

Real-world guidance: do not scatter hard-coded S3 paths throughout jobs. Use table names in PySpark transformations, and reserve physical storage paths for ingestion boundaries, external locations, or infrastructure configuration.

## How do you build an incremental PySpark pipeline?

An incremental pipeline processes only new or changed records. The usual pattern is to use a watermark, ingestion timestamp, source sequence number, or table change feed.

Simple batch watermark example:

In [ ]:
from pyspark.sql import functions as F

last_loaded_ts = (
    spark.table("main.ops.pipeline_watermarks")
    .where(F.col("pipeline_name") == "orders_silver")
    .select(F.max("loaded_until_ts").alias("ts"))
    .first()["ts"]
)

new_orders = (
    spark.table("main.bronze.orders")
    .where(F.col("ingested_at") > F.lit(last_loaded_ts))
)

Merge into the target table:

In [ ]:
new_orders.createOrReplaceTempView("new_orders")

spark.sql("""
merge into main.silver.orders as target
using new_orders as source
on target.order_id = source.order_id
when matched then update set *
when not matched then insert *
""")

Interview point: a reliable incremental job needs idempotency. If the job retries the same batch, the target should end in the same state, not duplicate data.

## How do you test PySpark transformations?

Keep business transformations as functions that accept and return DataFrames. Then create small test DataFrames and compare outputs.

Transformation:

In [ ]:
from pyspark.sql import functions as F

def add_order_date(df):
    return df.withColumn("order_date", F.to_date("order_ts"))

Test example:

In [ ]:
from datetime import datetime

input_df = spark.createDataFrame(
    [("o1", datetime(2026, 10, 4, 12, 30, 0))],
    ["order_id", "order_ts"],
)

actual = add_order_date(input_df).select("order_id", "order_date").collect()

assert str(actual[0]["order_date"]) == "2026-10-04"

For real projects, use `pytest`, fixtures for the Spark session, and small deterministic inputs. Test data should cover nulls, duplicate keys, late-arriving data, invalid records, and boundary timestamps.

## How do you handle bad records and data quality checks in PySpark?

Bad records should be isolated, measurable, and visible. Do not silently drop them unless the product owner agrees that loss is acceptable.

In [ ]:
from pyspark.sql import functions as F

orders = spark.table("main.bronze.orders")

valid_orders = orders.where(
    F.col("order_id").isNotNull()
    & F.col("customer_id").isNotNull()
    & (F.col("amount") >= 0)
)

invalid_orders = orders.subtract(valid_orders).withColumn(
    "rejected_at", F.current_timestamp()
)

invalid_orders.write.mode("append").saveAsTable("main.ops.rejected_orders")
valid_orders.write.mode("append").saveAsTable("main.silver.orders")

A more explicit rule-based pattern:

In [ ]:
checked = (
    orders
    .withColumn(
        "quality_error",
        F.when(F.col("order_id").isNull(), "missing_order_id")
         .when(F.col("amount") < 0, "negative_amount")
    )
)

checked.where("quality_error is null").write.mode("append").saveAsTable("main.silver.orders")
checked.where("quality_error is not null").write.mode("append").saveAsTable("main.ops.orders_quarantine")

Interview point: quality checks should happen before publishing gold tables. A dashboard should not discover broken data before the pipeline does.

## How do you use UDFs safely in PySpark?

Use Spark built-in functions first. Python UDFs often move data between the JVM and Python worker, which can be slower and harder for Spark to optimize.

Prefer this:

In [ ]:
from pyspark.sql import functions as F

cleaned = customers.withColumn(
    "email_domain",
    F.element_at(F.split(F.lower("email"), "@"), -1),
)

Instead of this:

In [ ]:
from pyspark.sql import functions as F, types as T

@F.udf(T.StringType())
def email_domain(email):
    if email is None or "@" not in email:
        return None
    return email.lower().split("@")[-1]

cleaned = customers.withColumn("email_domain", email_domain("email"))

If you need Python logic for vectorized numeric or ML work, consider Pandas UDFs, but measure performance and memory behavior.

In [ ]:
import pandas as pd
from pyspark.sql import functions as F, types as T

@F.pandas_udf(T.DoubleType())
def cap_score(score: pd.Series) -> pd.Series:
    return score.clip(lower=0.0, upper=1.0)

scored = users.withColumn("capped_score", cap_score("score"))

## How do you process streaming data with PySpark Structured Streaming?

Structured Streaming lets you express streaming logic with DataFrame APIs. Spark runs the query continuously or in micro-batches.

In [ ]:
from pyspark.sql import functions as F

events = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .schema("event_id string, user_id string, event_ts timestamp, event_type string")
    .load("s3://company-raw/events/")
)

clean_events = (
    events
    .withWatermark("event_ts", "30 minutes")
    .dropDuplicates(["event_id"])
)

query = (
    clean_events.writeStream
    .format("delta")
    .option("checkpointLocation", "s3://company-checkpoints/events_silver/")
    .trigger(processingTime="5 minutes")
    .toTable("main.silver.events")
)

Real-world points:

+ Checkpoint locations are required for fault tolerance.
+ Watermarks define how late data can arrive before Spark drops state.
+ Deduplication needs a stable event key.
+ Streaming writes must be idempotent or carefully checkpointed.

## How do you monitor and debug PySpark jobs?

Start with the Spark UI. It shows jobs, stages, tasks, shuffle read/write, spilled memory, input size, executor failures, and SQL plans.

Useful PySpark checks:

In [ ]:
df.explain("formatted")
print(df.rdd.getNumPartitions())
spark.conf.get("spark.sql.shuffle.partitions")

Common symptoms:

+ One task runs much longer than others: likely skew
+ Huge shuffle read/write: expensive join, aggregation, or repartition
+ Spilled memory: insufficient executor memory or too-wide shuffle
+ Many tiny tasks: small files or too many partitions
+ Driver out of memory: collecting too much data to the driver

For production jobs, log record counts at each pipeline boundary:

In [ ]:
input_count = orders.count()
output_count = valid_orders.count()

spark.createDataFrame(
    [("orders_silver", input_count, output_count)],
    ["pipeline_name", "input_count", "output_count"],
).write.mode("append").saveAsTable("main.ops.pipeline_metrics")

For deeper performance methods, see [Spark Tuning](spark-tuning.md).

## What should you say in a PySpark interview answer?

A strong PySpark interview answer should include the API, the execution model, and the production concern.

Example answer for "How do you deduplicate orders?":

In [ ]:
from pyspark.sql import Window, functions as F

w = Window.partitionBy("order_id").orderBy(F.col("source_updated_at").desc())

deduped = (
    spark.table("main.bronze.orders")
    .withColumn("rn", F.row_number().over(w))
    .where("rn = 1")
    .drop("rn")
)

Then explain:

+ `row_number` keeps one row per business key.
+ Ordering must use source metadata, not random ingestion order.
+ The result should be merged into the target for idempotency.
+ If one key is extremely hot, this window can skew and needs tuning.

That answer shows that you know PySpark syntax and the operational behavior behind it.

## Real world use cases for PySpark

### 1. Nightly batch ELT into a partitioned lake

Raw JSON lands in object storage, a Spark job cleans and types it, and the curated layer is written as partitioned Parquet for downstream engines.

In [ ]:
from pyspark.sql import functions as F

orders = spark.read.schema(order_schema).json("s3://acme-raw/orders/ds=2026-10-04/")

(
    orders.where("amount_cents > 0")
    .withColumn("amount", F.col("amount_cents") / 100)
    .repartition("order_date")            # one file family per day
    .sortWithinPartitions("country")      # tight min/max for skipping
    .write.mode("overwrite")
    .option("compression", "zstd")
    .partitionBy("order_date")
    .parquet("s3://acme-curated/orders/")
)

Why Spark: a few minutes of distributed CPU replaces hours of single-node scripting, and explicit schemas remove inference surprises. Enable Adaptive Query Execution (`spark.sql.adaptive.enabled=true`) so skewed joins and partition counts self-tune.

### 2. Structured Streaming from Kafka into a lakehouse table

A streaming job decodes Kafka events, applies the same transformations as the batch path, and merges micro-batches into an Iceberg or Delta table.

In [ ]:
from pyspark.sql import functions as F

(
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "broker1:9092")
    .option("subscribe", "orders")
    .option("startingOffsets", "earliest")
    .load()
    .select(F.from_json(F.col("value").cast("string"), order_schema).alias("o"))
    .select("o.*")
    .writeStream
    .foreachBatch(merge_into_lakehouse)          # idempotent MERGE per batch
    .option("checkpointLocation", "s3://acme-checkpoints/orders/")
    .trigger(processingTime="1 minute")
    .start()
)

Why Spark: the batch and streaming APIs share transformations, so there is one business-logic implementation. Checkpointing plus an idempotent merge gives end-to-end exactly-once.

### 3. Feature engineering for machine learning

Feature tables are built with broadcast joins to small dimensions and vectorized pandas UDFs instead of row-at-a-time Python UDFs.

In [ ]:
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.functions import pandas_udf

@pandas_udf("double")
def zscore(s: pd.Series) -> pd.Series:
    return (s - s.mean()) / s.std()

features = (
    events.join(F.broadcast(products), "product_id")   # small dimension
    .groupBy("customer_id")
    .agg(F.count("*").alias("orders_30d"), F.sum("amount").alias("spend_30d"))
    .withColumn("spend_z", zscore("spend_30d"))
)

features.write.mode("overwrite").parquet("s3://acme-features/customer/ds=2026-10-04/")

Why Spark: the same feature definitions can be applied to historical training data and to daily scoring runs, which reduces train/serve skew.

### 4. Log and event processing with messy input

Semi-structured logs break schema assumptions. Read permissively, quarantine bad rows, and evolve the schema deliberately.

In [ ]:
raw = (
    spark.read.option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_bad")
    .json("s3://acme-raw/logs/ds=2026-10-04/")
)

good = raw.filter("_bad is null").drop("_bad")
bad = raw.filter("_bad is not null")

good.write.mode("append").parquet("s3://acme-curated/logs/")
bad.write.mode("append").json("s3://acme-quarantine/logs/")

Why Spark: quarantine plus schema-on-read lets ingestion continue while the platform team fixes the producer, instead of failing the whole day.

### 5. Backfills and reprocessing without duplicating data

A bug fix requires reprocessing September. With dynamic partition overwrite, only the partitions you touch are replaced.

In [ ]:
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")

(
    reprocessed.withColumn("order_date", F.to_date("created_at"))
    .write.mode("overwrite")
    .partitionBy("order_date")
    .parquet("s3://acme-curated/orders/")
)

Why Spark: the same job that handles daily loads can reprocess a month, and dynamic overwrite makes the rerun idempotent at partition granularity - the property that makes backfills safe.

---

# Practice notebook: PySpark concepts

The cells below are from `notebooks/04_pyspark_concepts.ipynb` and are merged here so this Colab notebook is self-contained. Run them top to bottom; assertions confirm the expected behavior.

# 04 - PySpark Concepts, End to End

A hands-on tour of the PySpark concepts that come up most often in data-engineering interviews.

The notebook is deliberately granular: one idea per cell, a short explanation directly above
every guarded code cell, and a small self-check block of `assert`s after each cluster of
operations. Every Spark section has four parts:

1. **Concept** - the mental model, in plain language.
2. **Guarded code** - runnable PySpark that only executes when Spark is really available.
3. **Expected output** - what you should see, so the notebook still teaches without Spark.
4. **Interview note** - the trade-off a strong candidate mentions out loud.

## Prerequisites

- Notebook tooling: `uv sync --extra notebooks`
- Spark cells: `uv sync --extra notebooks --extra notebooks-spark`
- A **Java 17+** runtime on `PATH` (`java -version` must succeed).

PySpark talks to the JVM, so without Java `import pyspark` can succeed while every job fails.

## How to read this notebook without Spark

This machine may not have PySpark or Java installed. That is fine. The bootstrap cells probe
the environment and set `SPARK_AVAILABLE`. When it is `False`:

- every Spark cell prints `SKIPPED - install PySpark + Java 17 and re-run this cell`;
- the markdown **above** each Spark cell contains the expected output and the reasoning, so
  the concept is still fully covered;
- all `assert` statements live **inside** the Spark-available branch, so nothing fails just
  because Spark is missing.

## Concepts covered

SparkSession and local mode; DataFrame creation with an explicit schema; lazy evaluation;
`select`/`filter`/`withColumn`/`rename`/`cast`; null handling; aggregations
(`groupBy`/`agg`/`pivot`); joins and broadcast hints; window functions; UDFs vs built-ins;
arrays and `explode`; partitioning and Parquet partition pruning; caching; `explain()` plan
reading; Adaptive Query Execution and skew; small files and compaction; a Structured Streaming
sketch; Delta and Iceberg notes on Databricks.

## Running this notebook in Google Colab

The **Open in Colab** badge in `notebooks/README.md` opens this file straight from GitHub.
Colab provides a Java runtime but not PySpark, so run one setup cell first:

```python
!pip install pyspark
```

Then choose **Runtime -> Restart session** and run all cells. The bootstrap cells below
re-check the environment and turn the Spark cells on. If PySpark is unavailable the notebook
still runs: every Spark cell prints a SKIPPED message and the markdown above it explains the
expected result.

## Environment bootstrap: probe

**Expected output (no Spark here):**

```
PySpark installed : False
Java on PATH      : True
Java runnable     : False
Spark available   : False
Colab runtime     : False
```

`shutil.which("java")` only proves the binary exists: on some machines (e.g. macOS with only a
stub at `/usr/bin/java`) the executable is present but cannot start a JVM, so it is probed.

In [ ]:
import importlib.util
import shutil
import subprocess

HAS_PYSPARK = importlib.util.find_spec("pyspark") is not None
HAS_JAVA = shutil.which("java") is not None
try:
    IS_COLAB = importlib.util.find_spec("google.colab") is not None
except (ImportError, ValueError):  # `google` is not installed outside Colab
    IS_COLAB = False

# `shutil.which` only proves the binary exists; probe that it can start a JVM.
JAVA_RUNS = False
if HAS_JAVA:
    try:
        probe = subprocess.run(
            ["java", "-version"], capture_output=True, text=True, timeout=60
        )
        JAVA_RUNS = probe.returncode == 0
    except Exception:
        JAVA_RUNS = False

SPARK_AVAILABLE = HAS_PYSPARK and HAS_JAVA and JAVA_RUNS

print(f"PySpark installed : {HAS_PYSPARK}")
print(f"Java on PATH      : {HAS_JAVA}")
print(f"Java runnable     : {JAVA_RUNS}")
print(f"Spark available   : {SPARK_AVAILABLE}")
print(f"Colab runtime     : {IS_COLAB}")
print()


**Expected output (no Spark here):**

```
This notebook will run in SKIPPED mode: every Spark cell prints a
SKIPPED message, and the markdown above each cell still teaches the concept.

To enable the Spark cells:
  1. uv sync --extra notebooks --extra notebooks-spark
  2. Install a Java 17+ runtime (Temurin, or `brew install openjdk@17`)
  3. Make sure `java -version` works, then re-run this cell.
  - missing: a working Java 17+ runtime (the `java` on PATH cannot start)
  - missing: pyspark (the Python package)
```

**Interview note:** "it imports fine" is not "it runs". A quick capability probe up front
saves an hour of debugging a `JAVA_HOME`/`SparkContext` stack trace later - the same habit you
want in a production job that must fail fast on a broken runtime.

In [ ]:
if SPARK_AVAILABLE:
    import pyspark
    print(f"PySpark version   : {pyspark.__version__}")
    print("Spark cells below will execute for real.")
else:
    print("This notebook will run in SKIPPED mode: every Spark cell prints a")
    print("SKIPPED message, and the markdown above each cell still teaches the concept.")
    print()
    if IS_COLAB:
        print("Colab detected. To enable the Spark cells here:")
        print("  1. Add a cell containing:  !pip install pyspark")
        print("  2. Run it, then choose Runtime -> Restart session")
        print("  3. Run all cells again (Colab already provides Java).")
    else:
        print("To enable the Spark cells:")
        print("  1. uv sync --extra notebooks --extra notebooks-spark")
        print("  2. Install a Java 17+ runtime (Temurin, or `brew install openjdk@17`)")
        print("  3. Make sure `java -version` works, then re-run this cell.")
    if not HAS_PYSPARK:
        print("  - missing: pyspark (the Python package)")
    if not HAS_JAVA:
        print("  - missing: a `java` executable on PATH")
    elif not JAVA_RUNS:
        print("  - missing: a working Java 17+ runtime (the `java` on PATH cannot start)")


## 1. SparkSession and local mode

### Concept

`SparkSession` is the single entry point to Spark SQL and DataFrames. It wraps a
`SparkContext` (the low-level RDD scheduler) plus the SQL catalog and configuration.

### Why `local[*]`

`master("local[*]")` runs the whole cluster - driver and one executor - inside this JVM,
using one core per CPU for parallelism.

In a cluster you drop `.master(...)` and let the cluster manager (YARN, Kubernetes,
Databricks) supply it. Hard-coding a master is the classic bug that makes a job run on one
node in production.

Key configs used here:

- `spark.sql.shuffle.partitions` - partitions produced by a shuffle (default 200).
- `spark.ui.enabled=false` - skip the web UI in a notebook.

### Expected output (with Spark)

```
Spark version : 3.5.x
Master        : local[*]
App name      : de-interview-pyspark-concepts
```

### Interview note

Describe the driver/executor split here. The driver plans work and collects results; executors
run tasks and hold cached blocks. `local[*]` collapses both into one process, which is why it
is a dev tool, not a deployment topology.

In [ ]:
if SPARK_AVAILABLE:
    from pyspark.sql import SparkSession

    spark = (
        SparkSession.builder
        .appName("de-interview-pyspark-concepts")
        .master("local[*]")
        .config("spark.sql.shuffle.partitions", "8")
        .config("spark.ui.enabled", "false")
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("ERROR")

    print("Spark version :", spark.version)
    print("Master        :", spark.sparkContext.master)
    print("App name      :", spark.sparkContext.appName)
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### Expected output (with Spark)

```
self-check 1 passed: local session, correct app name
```

**Check yourself.** The session must be local and named exactly as requested.

In [ ]:
if SPARK_AVAILABLE:
    # --- self-check 1 ---------------------------------------------------
    assert spark.sparkContext.master.startswith("local"), spark.sparkContext.master
    assert spark.sparkContext.appName == "de-interview-pyspark-concepts"
    print("self-check 1 passed: local session, correct app name")
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


## 2. DataFrames from Python data with an explicit schema

### Concept

`spark.createDataFrame(data, schema)` turns local Python data into a distributed DataFrame.

### Why an explicit schema

Passing an explicit `StructType` rather than letting Spark infer types is the production habit:
inference reads the data twice, guesses `string` for dates and numbers, and silently changes
nullability. Explicit schemas also document the contract and fail fast when the upstream shape
changes.

`LongType` / `DoubleType` / `StringType` map to `bigint` / `double` / `string` in the
DataFrame. The third field of `StructField` is `nullable`.

### Expected output (with Spark) - schema

```
root
 |-- order_id: long (nullable = false)
 |-- customer_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- order_ts: string (nullable = true)
```

### Expected output (with Spark) - the frame

```
+--------+-----------+------+---------+-------------------+
|order_id|customer_id|amount|status   |order_ts           |
+--------+-----------+------+---------+-------------------+
|1       |c1         |120.5 |paid     |2026-10-01 10:00:00|
|2       |c2         |80.0  |paid     |2026-10-01 11:30:00|
|3       |c3         |200.0 |shipped  |2026-10-02 09:15:00|
|4       |c1         |15.75 |cancelled|2026-10-02 12:00:00|
|5       |c4         |60.0  |paid     |2026-10-03 08:45:00|
|6       |c2         |30.0  |returned |2026-10-03 16:20:00|
+--------+-----------+------+---------+-------------------+
```

### Interview note

Mention schema enforcement and corrupt-record handling together:
`mode=PERMISSIVE/FAILFAST/DROPMALFORMED` on the reader, and why `FAILFAST` in a curated layer is
usually right. A schema inferred in dev and drifted in prod is a top cause of silent data
corruption.

In [ ]:
if SPARK_AVAILABLE:
    from pyspark.sql.types import (
        DoubleType,
        LongType,
        StringType,
        StructField,
        StructType,
    )

    orders_schema = StructType([
        StructField("order_id", LongType(), False),
        StructField("customer_id", StringType(), True),
        StructField("amount", DoubleType(), True),
        StructField("status", StringType(), True),
        StructField("order_ts", StringType(), True),
    ])
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### Building the rows

`orders_data` is a plain Python list of tuples; it is not distributed until
`createDataFrame` sees it.

**Expected output (with Spark):** the same six rows shown in the table above.

In [ ]:
if SPARK_AVAILABLE:
    orders_data = [
        (1, "c1", 120.50, "paid", "2026-10-01 10:00:00"),
        (2, "c2", 80.00, "paid", "2026-10-01 11:30:00"),
        (3, "c3", 200.00, "shipped", "2026-10-02 09:15:00"),
        (4, "c1", 15.75, "cancelled", "2026-10-02 12:00:00"),
        (5, "c4", 60.00, "paid", "2026-10-03 08:45:00"),
        (6, "c2", 30.00, "returned", "2026-10-03 16:20:00"),
    ]

    orders = spark.createDataFrame(orders_data, schema=orders_schema)
    orders.printSchema()
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### Inspecting the frame

`show(truncate=False)` reprints the table without cutting long column values.

**Expected output (with Spark):** the six-row table above.

In [ ]:
if SPARK_AVAILABLE:
    orders.show(truncate=False)
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### Expected output (with Spark)

```
self-check 2 passed: 6 rows, amount typed double, order_id non-nullable
```

**Check yourself.** Row count, the declared type of `amount`, and the nullability of
`order_id` are all part of the schema contract.

In [ ]:
if SPARK_AVAILABLE:
    # --- self-check 2 ---------------------------------------------------
    assert orders.count() == 6
    assert orders.schema["amount"].dataType == DoubleType()
    assert orders.schema["order_id"].nullable is False
    print("self-check 2 passed: 6 rows, amount typed double, order_id non-nullable")
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


## 3. Lazy evaluation: transformations vs actions

### Concept

A **transformation** (`filter`, `withColumn`, `select`, `join`, `groupBy`) only *describes* new
work and returns a new immutable DataFrame. Nothing is scheduled.

An **action** (`count`, `show`, `collect`, `write`, `take`, `foreach`) asks for a concrete
result; only then does Catalyst optimise the whole chain into stages and tasks and the
scheduler launch them.

### Why it matters

This is why a notebook cell that builds five DataFrames finishes instantly, and the cell with a
single `.count()` is the one that costs money. It is also why `withColumn` chains compile into
one scan rather than five passes.

### Expected output (with Spark)

```
After two transformations, no job has been submitted yet.
isStreaming = False
paid rows: 3
```

### Interview note

The follow-up is always "so when does a job run?". Answer: at the action, and one action = one
job = one fresh pass over the source unless you cached or reused a shuffled/saved result.

In [ ]:
if SPARK_AVAILABLE:
    from pyspark.sql import functions as F

    # Transformation 1 - lazy.
    paid_only = orders.filter(F.col("status") == "paid")
    # Transformation 2 - still lazy, still nothing executed.
    with_tax = paid_only.withColumn("amount_with_tax", F.round(F.col("amount") * 1.08, 2))

    print("After two transformations, no job has been submitted yet.")
    print("isStreaming =", with_tax.isStreaming)
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### Reading the plan without running it

`explain(True)` prints the parsed, analysed and optimised plans; it reads no data.

**Expected output (with Spark):** a logical/physical plan tree for the filter-plus-withColumn
chain.

In [ ]:
if SPARK_AVAILABLE:
    with_tax.explain(True)
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### The action is what triggers a job

**Expected output (with Spark):**

```
paid rows: 3
```

In [ ]:
if SPARK_AVAILABLE:
    paid_rows = with_tax.count()  # <-- action: this is what triggers a job
    print("paid rows:", paid_rows)
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### Expected output (with Spark)

```
self-check 1b passed: the action returned 3 paid rows
```

**Check yourself.** The count only exists because the action above ran.

In [ ]:
if SPARK_AVAILABLE:
    assert paid_rows == 3
    print("self-check 1b passed: the action returned 3 paid rows")
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


## 4. `select`, `filter`, `withColumn`, `rename`, `cast`

### Concept

These are the everyday column transformations and all are **narrow** (no data exchange between
executors):

- `select` / `drop` choose columns - push the projection down so Parquet reads fewer bytes.
- `filter` / `where` are aliases; predicates are pushed to the scan when possible.
- `withColumn` replaces or adds a column from an expression.
- `withColumnRenamed` renames; `alias` does it inline in a `select`.
- `cast` changes the type; do it explicitly and early, and know that a bad cast yields `null`
  (masking errors) instead of raising.

### Expected output (with Spark)

```
root
 |-- order_id: long (nullable = false)
 |-- cust_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- order_ts: timestamp (nullable = true)
 |-- amount_cents: long (nullable = true)

5 rows, columns: ['order_id', 'cust_id', 'amount', 'order_ts', 'amount_cents']
amount_cents first row = 12050
```

### Interview note

Say "narrow transformation" and explain why it matters: narrow steps stay inside a partition,
so they need no shuffle. Wide steps (`groupBy`, `join`, `distinct`, `repartition`, window with a
different partitioning) force an exchange and dominate cost.

### Step 1 - project and filter

**Expected output (with Spark):** five rows, the union of statuses `paid` and `shipped`.

In [ ]:
if SPARK_AVAILABLE:
    selected = (
        orders
        .select("order_id", "customer_id", "amount", "status", "order_ts")
        .filter(F.col("status").isin("paid", "shipped"))
    )
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### Step 2 - derive and cast columns

`to_timestamp` parses the string into a `timestamp`; multiplying by 100 and casting to `long`
turns dollars into integer cents.

In [ ]:
if SPARK_AVAILABLE:
    derived = (
        selected
        .withColumn("order_ts", F.to_timestamp("order_ts"))
        .withColumn("amount_cents", (F.col("amount") * 100).cast("long"))
    )
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### Step 3 - rename and drop

`withColumnRenamed("customer_id", "cust_id")` renames in place; `drop("status")` removes a
column. Both keep the frame narrow.

In [ ]:
if SPARK_AVAILABLE:
    transformed = derived.withColumnRenamed("customer_id", "cust_id").drop("status")
    transformed.printSchema()
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### Evaluate

**Expected output (with Spark):**

```
5 rows, columns: ['order_id', 'cust_id', 'amount', 'order_ts', 'amount_cents']
amount_cents first row = 12050
```

In [ ]:
if SPARK_AVAILABLE:
    rows = transformed.collect()
    print(len(rows), "rows, columns:", transformed.columns)
    print("amount_cents first row =", rows[0]["amount_cents"])
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### Expected output (with Spark)

```
self-check 3 passed: types, rename and row count all correct
```

**Check yourself.** `count()` is an action, so this cell also materialises the chain.

In [ ]:
if SPARK_AVAILABLE:
    # --- self-check 3 ---------------------------------------------------
    assert transformed.count() == 5
    assert "cust_id" in transformed.columns and "customer_id" not in transformed.columns
    assert dict(transformed.dtypes)["amount_cents"] == "bigint"
    assert dict(transformed.dtypes)["order_ts"] == "timestamp"
    print("self-check 3 passed: types, rename and row count all correct")
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


## 5. Null handling

### Concept

Spark SQL is null-aware: `null == null` is `null` (not true), so use `isNull` / `isNotNull` /
`eqNullSafe` (`<=>`) rather than `==`.

`F.col("x") == None` does **not** work - the comparison yields null and filters everything.

### The toolkit

- `isNull()`, `isNotNull()` for predicates;
- `na.drop(subset=[...])` to remove rows, `na.fill({...})` to substitute values;
- `coalesce(a, b, lit(default))` to pick the first non-null expression;
- `when(...).otherwise(...)` for conditional logic (handles null explicitly);
- `nullif` to turn a sentinel value back into a null.

### Expected output (with Spark)

```
raw rows: 4
rows with any of amount/status null: 3
after fill+drop: 3 rows, 0 null amounts, 0 null statuses
coalesce produced 2 'unknown' statuses
```

### Interview note

Tie nulls to the metric. `avg` ignores nulls (denominator excludes them), `sum` of all-null is
null while `sum` with 0s is 0, and `count(col)` skips nulls while `count(*)` does not.
Interviewers use exactly this to test whether you trust a dashboard number.

In [ ]:
if SPARK_AVAILABLE:
    nulls = spark.createDataFrame(
        [
            (1, "c1", 120.5, None),
            (2, "c2", None, "paid"),
            (3, None, 50.0, "paid"),
            (4, "c4", 0.0, None),
        ],
        "order_id long, customer_id string, amount double, status string",
    )
    nulls.show()
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### Counting the holes

**Expected output (with Spark):**

```
raw rows: 4
rows with any of amount/status null: 3
```

In [ ]:
if SPARK_AVAILABLE:
    missing = nulls.filter(F.col("amount").isNull() | F.col("status").isNull())
    print("raw rows:", nulls.count())
    print("rows with any of amount/status null:", missing.count())
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### Fill and drop

`na.fill` substitutes values for nulls, then `na.drop(subset=[...])` removes rows whose key
column is still null.

**Expected output (with Spark):**

```
after fill+drop: 3 rows, 0 null amounts, 0 null statuses
```

In [ ]:
if SPARK_AVAILABLE:
    filled = nulls.na.fill({"amount": 0.0, "status": "unknown"}).na.drop(subset=["customer_id"])
    filled.show()
    print(
        "after fill+drop:",
        filled.count(),
        "rows,",
        filled.filter(F.col("amount").isNull()).count(),
        "null amounts,",
        filled.filter(F.col("status").isNull()).count(),
        "null statuses",
    )
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### `coalesce` for a default

`coalesce` returns the first non-null argument, so it is the clean way to supply a fallback.

**Expected output (with Spark):**

```
coalesce produced 2 'unknown' statuses
```

In [ ]:
if SPARK_AVAILABLE:
    coalesced = nulls.withColumn("status_clean", F.coalesce(F.col("status"), F.lit("unknown")))
    print("coalesce produced", coalesced.filter(F.col("status_clean") == "unknown").count(), "'unknown' statuses")
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### Expected output (with Spark)

```
self-check 4 passed: null semantics, fill/drop and coalesce all correct
```

In [ ]:
if SPARK_AVAILABLE:
    # --- self-check 4 ---------------------------------------------------
    assert missing.count() == 3
    assert filled.count() == 3
    assert filled.filter(F.col("amount").isNull()).count() == 0
    assert filled.filter(F.col("status").isNull()).count() == 0
    assert coalesced.filter(F.col("status_clean") == "unknown").count() == 2
    print("self-check 4 passed: null semantics, fill/drop and coalesce all correct")
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


## 6. Aggregations: `groupBy` / `agg` / `pivot`

### Concept

`groupBy(...).agg(...)` is a **wide** transformation: rows with the same key must land in the
same partition, so Spark performs a shuffle (a hash exchange) unless the data is already
co-partitioned.

### What a shuffle costs

Costs to name in an interview: shuffle write + read, disk spill, and the
`spark.sql.shuffle.partitions` fan-out.

### Pivot

`pivot` rotates a low-cardinality column into columns. It must first buffer the distinct pivot
values, so Spark asks you to supply them explicitly when you know them -
`.pivot("product", ["gadget", "widget"])` - which avoids an extra pass over the data.

### Expected output (with Spark)

```
+------+-------+------+-------+
|region|revenue|orders|avg_qty|
+------+-------+------+-------+
|north |105.0  |2     |6.5    |
|south |125.0  |2     |6.0    |
+------+-------+------+-------+

+------+------+------+
|region|gadget|widget|
+------+------+------+
|north |30.0  |25.0  |
|south |70.0  |12.5  |
+------+------+------+
```

The pivot table is revenue by product: widget = 10*2.5 = 25 (north) and 5*2.5 = 12.5 (south);
gadget = 3*10 = 30 (north) and 7*10 = 70 (south).

### Interview note

Follow up `groupBy` with partition sizing. Too few post-shuffle partitions and one task becomes
the bottleneck (skew); too many and you pay scheduler and small-file overhead. AQE can coalesce
them for you - see section 14.

### Build the input

**Expected output (with Spark):** a four-row frame of sales.

In [ ]:
if SPARK_AVAILABLE:
    sales = spark.createDataFrame(
        [
            ("2026-10-01", "north", "widget", 10, 2.5),
            ("2026-10-01", "south", "widget", 5, 2.5),
            ("2026-10-02", "north", "gadget", 3, 10.0),
            ("2026-10-02", "south", "gadget", 7, 10.0),
        ],
        "sale_date string, region string, product string, qty int, unit_price double",
    )
    sales.show()
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### `groupBy().agg()`

Three aggregates in one pass: a sum, a count and an average.

**Expected output (with Spark):** the `per_region` table above.

In [ ]:
if SPARK_AVAILABLE:
    per_region = (
        sales
        .withColumn("revenue", F.col("qty") * F.col("unit_price"))
        .groupBy("region")
        .agg(
            F.sum("revenue").alias("revenue"),
            F.count("*").alias("orders"),
            F.avg("qty").alias("avg_qty"),
        )
    )
    per_region.show()
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### `pivot`

An explicit pivot-value list (`["gadget", "widget"]`) avoids the extra pass that Spark would
otherwise need to discover the distinct values.

**Expected output (with Spark):** the `region/gadget/widget` table above.

In [ ]:
if SPARK_AVAILABLE:
    pivoted = (
        sales
        .withColumn("revenue", F.col("qty") * F.col("unit_price"))
        .groupBy("region")
        .pivot("product", ["gadget", "widget"])
        .agg(F.sum("revenue"))
    )
    pivoted.show()
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### Expected output (with Spark)

```
self-check 5 passed: aggregation types and pivot values correct
```

In [ ]:
if SPARK_AVAILABLE:
    # --- self-check 5 ---------------------------------------------------
    assert per_region.count() == 2
    assert dict(per_region.dtypes)["revenue"] == "double"
    assert set(pivoted.columns) == {"region", "gadget", "widget"}
    assert pivoted.filter(F.col("region") == "south").collect()[0]["gadget"] == 70.0
    print("self-check 5 passed: aggregation types and pivot values correct")
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


## 7. Joins: inner, left, broadcast hints, and why shuffle happens

### Concept

A join keys two datasets on a common column. `how=` selects the semantics:

- `inner` keeps matches only;
- `left` keeps every left row and null-fills the right side;
- `left_anti` keeps left rows with **no** match (great for orphan detection);
- `left_semi` keeps left rows that match (great for existence checks).

### Why a shuffle?

In a standard sort-merge/hash join, matching keys must be in the same partition, so Spark
repartitions both sides by the join key: a full network exchange.

### Broadcast join

If one side is small (default limit `spark.sql.autoBroadcastJoinThreshold` = 10 MB,
configurable), Spark ships that whole side to every executor and joins locally - no shuffle of
the big table. You force it with `F.broadcast(small_df)`, and the hint is ignored if the side
exceeds the threshold. `left_anti` also avoids the wide shuffle when it can be turned into a
broadcast anti-join.

### Expected output (with Spark)

```
inner rows: 2   (o3 has customer c9, which is not in customers)
left rows : 3
left rows with no matching customer: 1
broadcast join rows: 3
```

### Interview note

The broadcast limit is about memory, not just time. Every executor must hold the small side; a
4 GB "small" dimension table in `local[*]` or on a small executor causes OOM. Also mention that
a **skewed** join key (one key = 40% of rows) defeats the shuffle join regardless of sizing -
the fix is salting or AQE skew join.

### Build the inputs

`customers` is the small dimension; `small_orders` has one orphan row (`c9`).

In [ ]:
if SPARK_AVAILABLE:
    customers = spark.createDataFrame(
        [("c1", "Ada"), ("c2", "Grace"), ("c3", "Linus")],
        "customer_id string, name string",
    )
    small_orders = spark.createDataFrame(
        [("o1", "c1", 10.0), ("o2", "c2", 20.0), ("o3", "c9", 5.0)],
        "order_id string, customer_id string, amount double",
    )
    customers.show()
    small_orders.show()
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### Inner and left joins

**Expected output (with Spark):**

```
inner rows: 2   (o3 has customer c9, which is not in customers)
left rows : 3
```

In [ ]:
if SPARK_AVAILABLE:
    inner = small_orders.join(customers, on="customer_id", how="inner")
    left = small_orders.join(customers, on="customer_id", how="left")

    print("inner rows:", inner.count(), "  (o3 has customer c9, which is not in customers)")
    print("left rows :", left.count())
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### Finding the orphan

`left` null-fills the unmatched right side, so a null `name` marks the orphan.

**Expected output (with Spark):**

```
left rows with no matching customer: 1
```

In [ ]:
if SPARK_AVAILABLE:
    print("left rows with no matching customer:", left.filter(F.col("name").isNull()).count())
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### Broadcast hint

`broadcast(customers)` forces the small side to be shipped to every executor.
`explain()` should show `BroadcastHashJoin` instead of `SortMergeJoin`.

**Expected output (with Spark):**

```
broadcast join rows: 3
```

In [ ]:
if SPARK_AVAILABLE:
    from pyspark.sql.functions import broadcast

    with_hint = small_orders.join(broadcast(customers), on="customer_id", how="left")
    with_hint.explain()  # look for BroadcastHashJoin in the physical plan
    print("broadcast join rows:", with_hint.count())
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### `left_anti` for set difference

`left_anti` finds orphans without materialising right-hand columns.

**Expected output (with Spark):** one row, `o3`.

In [ ]:
if SPARK_AVAILABLE:
    orphans = small_orders.join(customers, on="customer_id", how="left_anti")
    orphans.show()
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### Expected output (with Spark)

```
self-check 6 passed: join semantics and broadcast hint correct
```

In [ ]:
if SPARK_AVAILABLE:
    # --- self-check 6 ---------------------------------------------------
    assert inner.count() == 2
    assert left.count() == 3
    assert left.filter(F.col("name").isNull()).count() == 1
    assert with_hint.count() == 3
    assert orphans.collect()[0]["order_id"] == "o3"
    print("self-check 6 passed: join semantics and broadcast hint correct")
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


## 8. Window functions

### Concept

A window function computes over a set of rows related to the current row **without collapsing
them** the way `groupBy` does. You define a `Window`:

- `partitionBy(...)` - the groups (this is what forces a shuffle);
- `orderBy(...)` - the ordering inside each group;
- `rowsBetween(...)` / `rangeBetween(...)` - the frame for aggregations.

### The four workhorses

- `row_number()` - unique 1..n per partition; the standard **dedupe** idiom (`rn = 1` keeps
  exactly one row per key even for exact ties).
- `rank()` / `dense_rank()` - leave gaps / do not leave gaps on ties, so ties produce the same
  rank and more than one "rank 1".
- `sum(...).over(unboundedPreceding..currentRow)` - running total.
- `lag()` / `lead()` - previous / next value, e.g. time between events.

### Expected output (with Spark)

```
row_number dedupe keeps: 3 rows (one per customer)
c2 winner amount: 99.0
rows with rank() == 1     : 4   (the tied pair shares rank 1)
c2 rank() values: [1, 1, 3]   dense_rank() values: [1, 1, 2]
running total for c1: 175.0
rows where lag() is null: 3   (one first row per customer)
```

Note the tie: `rank()` jumps to 3 after the tied pair, `dense_rank()` continues at 2. That is
the difference interviewers probe for.

### Interview note

"dedupe with `row_number` over the natural key ordered by `updated_at desc`, then `rn = 1`" is
the canonical CDC/upsert answer. Mention the tie-break: put a deterministic second sort key in
the `orderBy`, or the "latest" row is non-deterministic across runs.

### Build the events

`c2` has an exact duplicate row, which creates a real tie for `rank()`.

**Expected output (with Spark):** the seven-row frame.

In [ ]:
if SPARK_AVAILABLE:
    from pyspark.sql.window import Window

    events = spark.createDataFrame(
        [
            ("c1", "2026-10-01", 100.0),
            ("c1", "2026-10-03", 50.0),
            ("c1", "2026-10-02", 25.0),
            ("c2", "2026-10-01", 10.0),
            ("c2", "2026-10-01", 99.0),
            ("c2", "2026-10-01", 99.0),  # exact duplicate -> a real tie
            ("c3", "2026-10-05", 7.0),
        ],
        "customer_id string, order_date string, amount double",
    )
    events.show()
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### The window spec

Latest first, and `amount` desc as a deterministic tie-break so the "latest" row never depends
on run order.

In [ ]:
if SPARK_AVAILABLE:
    by_customer_latest = Window.partitionBy("customer_id").orderBy(
        F.col("order_date").desc(), F.col("amount").desc()
    )
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### Dedupe with `row_number`

**Expected output (with Spark):**

```
row_number dedupe keeps: 3 rows (one per customer)
c2 winner amount: 99.0
```

In [ ]:
if SPARK_AVAILABLE:
    deduped = (
        events
        .withColumn("rn", F.row_number().over(by_customer_latest))
        .filter(F.col("rn") == 1)
        .drop("rn")
    )
    print("row_number dedupe keeps:", deduped.count(), "rows (one per customer)")
    c2_amount = deduped.filter(F.col("customer_id") == "c2").collect()[0]["amount"]
    print("c2 winner amount:", c2_amount)
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### `rank()` vs `dense_rank()`

**Expected output (with Spark):**

```
rows with rank() == 1     : 4   (the tied pair shares rank 1)
c2 rank() values: [1, 1, 3]   dense_rank() values: [1, 1, 2]
```

In [ ]:
if SPARK_AVAILABLE:
    ranked = events.withColumn("rk", F.rank().over(by_customer_latest))
    dense = events.withColumn("drk", F.dense_rank().over(by_customer_latest))
    print("rows with rank() == 1     :", ranked.filter(F.col("rk") == 1).count(), "  (the tied pair shares rank 1)")
    c2_ranks = [r["rk"] for r in ranked.filter(F.col("customer_id") == "c2").orderBy("rk").collect()]
    c2_dense = [r["drk"] for r in dense.filter(F.col("customer_id") == "c2").orderBy("drk").collect()]
    print("c2 rank() values:", c2_ranks, "  dense_rank() values:", c2_dense)
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### Running total

A frame from `unboundedPreceding` to `currentRow` turns `sum` into a running total.

**Expected output (with Spark):**

```
running total for c1: 175.0
```

In [ ]:
if SPARK_AVAILABLE:
    running_window = (
        Window.partitionBy("customer_id")
        .orderBy("order_date")
        .rowsBetween(Window.unboundedPreceding, Window.currentRow)
    )
    running = events.withColumn("running_amount", F.sum("amount").over(running_window))
    c1_total = running.filter(F.col("customer_id") == "c1").agg(F.max("running_amount")).collect()[0][0]
    print("running total for c1:", c1_total)
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### `lag` for the previous value

**Expected output (with Spark):**

```
rows where lag() is null: 3   (one first row per customer)
```

In [ ]:
if SPARK_AVAILABLE:
    lagged = events.withColumn(
        "prev_amount",
        F.lag("amount").over(Window.partitionBy("customer_id").orderBy("order_date")),
    )
    print("rows where lag() is null:", lagged.filter(F.col("prev_amount").isNull()).count())
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### Expected output (with Spark)

```
self-check 7 passed: dedupe, rank tie semantics, running total, lag
```

In [ ]:
if SPARK_AVAILABLE:
    # --- self-check 7 ---------------------------------------------------
    assert deduped.count() == 3
    assert c2_amount == 99.0
    assert ranked.filter(F.col("rk") == 1).count() == 4
    assert c2_ranks == [1, 1, 3]
    assert c2_dense == [1, 1, 2]
    assert c1_total == 175.0
    assert lagged.filter(F.col("prev_amount").isNull()).count() == 3
    print("self-check 7 passed: dedupe, rank tie semantics, running total, lag")
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


## 9. UDFs vs built-in functions

### Concept

A Python **UDF** ships a serialised Python function to each executor task, runs it **row by
row**, and converts every value between the JVM and the Python worker. That breaks Catalyst's
ability to optimise the expression and adds per-row pickling overhead - often 2-10x slower than
an equivalent built-in, and it blocks whole-stage code generation (Tungsten).

### Rules of thumb

1. Prefer `pyspark.sql.functions` (`when`, `regexp_extract`, `to_date`, `array_*`, ...).
2. If you truly need custom logic, prefer a **pandas UDF** (`@pandas_udf`) so work is vectorised
   per batch, or `F.udf` only for small, non-hot paths.
3. Never do network/DB calls inside a UDF - that is per-row I/O on every executor.

The example below computes the same tier with both approaches. Both give identical results; the
built-in is the one you should ship.

### Expected output (with Spark)

```
+-------+---------+
|amount |tier_udf |
+-------+---------+
|120.5  |high     |
|80.0   |mid      |
|200.0  |high     |
|15.75  |low      |
|60.0   |mid      |
|30.0   |low      |
+-------+---------+

UDF 'high' rows   : 2
Built-in 'high' rows: 2
```

### Interview note

The expected answer is "UDFs are a code smell in hot paths". Add the escape hatch: pandas UDFs
for vectorisation and, on Databricks, a Scala/Java UDF or `transform`/higher-order SQL
functions for zero-copy JVM execution.

### Define the UDF

**Expected output (with Spark):** nothing yet - the UDF is only registered here.

In [ ]:
if SPARK_AVAILABLE:
    from pyspark.sql.types import StringType

    def _tier(amount):
        if amount is None:
            return "unknown"
        if amount >= 100:
            return "high"
        if amount >= 50:
            return "mid"
        return "low"

    tier_udf = F.udf(_tier, StringType())
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### The built-in equivalent

`when(...).otherwise(...)` expresses the same branching as an optimisable expression.

In [ ]:
if SPARK_AVAILABLE:
    with_udf = orders.withColumn("tier_udf", tier_udf(F.col("amount")))
    with_builtin = orders.withColumn(
        "tier_builtin",
        F.when(F.col("amount") >= 100, "high")
        .when(F.col("amount") >= 50, "mid")
        .otherwise("low"),
    )
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### Same answer, different cost

**Expected output (with Spark):**

```
UDF 'high' rows   : 2
Built-in 'high' rows: 2
```

In [ ]:
if SPARK_AVAILABLE:
    with_udf.select("amount", "tier_udf").show()
    print("UDF 'high' rows   :", with_udf.filter(F.col("tier_udf") == "high").count())
    print("Built-in 'high' rows:", with_builtin.filter(F.col("tier_builtin") == "high").count())
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### Expected output (with Spark)

```
self-check 8 passed: UDF and built-in agree
```

In [ ]:
if SPARK_AVAILABLE:
    # --- self-check 8 ---------------------------------------------------
    # Same answer, but the built-in version can be optimised and code-generated.
    assert with_udf.filter(F.col("tier_udf") == "high").count() == 2
    assert with_builtin.filter(F.col("tier_builtin") == "high").count() == 2
    print("self-check 8 passed: UDF and built-in agree")
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


## 10. Arrays and `explode`

### Concept

Semi-structured values (JSON lists, multi-valued tags) are natural as `array<T>` columns.

### The core operators

- `explode(col)` - one output row per element, **dropping null/empty arrays**.
- `explode_outer(col)` - same, but keeps a row with `null` when the array is null/empty.
- `array(...)`, `array_contains`, `array_distinct`, `size`, `concat`, `collect_list`
  for building and querying arrays;
- `posexplode` when the index matters.

`collect_list`/`collect_set` in a `groupBy` are the reverse operation: rows back into an array.

### Expected output (with Spark)

```
explode rows      : 3   (u1 has 2 tags, u2 has 1, empty/null arrays vanish)
explode_outer rows: 5   (u3 and u4 appear with a null tag)
tables with tag 'new': 4
```

### Interview note

`explode` is a common hidden row multiplier - it inflates row counts and therefore file sizes
and shuffle volume. Mention the `explode_outer` choice explicitly: silently dropping rows with
empty arrays is a classic correctness bug in event pipelines.

### Build the input

`u3` has an empty array and `u4` has `null`.

**Expected output (with Spark):** the four-row frame with a `tags` array column.

In [ ]:
if SPARK_AVAILABLE:
    users = spark.createDataFrame(
        [
            ("u1", ["spark", "sql"]),
            ("u2", ["python"]),
            ("u3", []),
            ("u4", None),
        ],
        "user_id string, tags array<string>",
    )
    users.show(truncate=False)
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### `explode` vs `explode_outer`

**Expected output (with Spark):**

```
explode rows      : 3   (u1 has 2 tags, u2 has 1, empty/null arrays vanish)
explode_outer rows: 5   (u3 and u4 appear with a null tag)
```

In [ ]:
if SPARK_AVAILABLE:
    exploded = users.select("user_id", F.explode("tags").alias("tag"))
    exploded_outer = users.select("user_id", F.explode_outer("tags").alias("tag"))
    print("explode rows      :", exploded.count(), "  (u1 has 2 tags, u2 has 1, empty/null arrays vanish)")
    print("explode_outer rows:", exploded_outer.count(), "  (u3 and u4 appear with a null tag)")
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### Building arrays with `concat` / `array_distinct`

`coalesce(tags, array())` replaces a null array with an empty one so `concat` always works.

**Expected output (with Spark):**

```
tables with tag 'new': 4
```

In [ ]:
if SPARK_AVAILABLE:
    augmented = users.withColumn(
        "tags",
        F.array_distinct(
            F.concat(F.coalesce(F.col("tags"), F.array().cast("array<string>")), F.array(F.lit("new")))
        ),
    )
    augmented.show(truncate=False)
    print("tables with tag 'new':", augmented.filter(F.array_contains(F.col("tags"), "new")).count())
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### Expected output (with Spark)

```
self-check 9 passed: explode row counts and array ops correct
```

In [ ]:
if SPARK_AVAILABLE:
    # --- self-check 9 ---------------------------------------------------
    assert exploded.count() == 3
    assert exploded_outer.count() == 5
    assert augmented.filter(F.array_contains(F.col("tags"), "new")).count() == 4
    print("self-check 9 passed: explode row counts and array ops correct")
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


## 11. Partitioning and Parquet write/read with partition pruning

### Concept

`partitionBy` on a **low-cardinality** column writes the data into a directory per value
(`order_date=2026-10-01/part-....parquet`) and removes that column from the files themselves.

A later query that filters on the partition column can then read only the matching directories -
**partition pruning** - which is the single biggest I/O win in most lakehouse jobs.

### Parquet's extra wins

Parquet adds column pruning and predicate pushdown inside each file via its footer metadata,
plus compression and encoding.

### Choosing partition keys

Thousands of tiny `date=.../hour=.../user_id=...` directories is "over-partitioning" and is
worse than not partitioning at all. Target roughly 100 MB-1 GB per partition file.

### Expected output (with Spark)

```
root
 |-- order_id: long (nullable = false)
 |-- customer_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- order_ts: string (nullable = true)
 |-- order_date: date (nullable = true)

rows read back        : 6
rows for 2026-10-01   : 2
partition directories : ['order_date=2026-10-01', 'order_date=2026-10-02', 'order_date=2026-10-03']
```

### Interview note

Mention pruning **and** its limits: filtering on a non-partition column still scans every file
(it only skips row groups via pushdown). This is exactly why low-cardinality partition keys plus
Parquet stats are the standard lakehouse layout, and why tools like Delta/Iceberg add file-level
min/max statistics.

### Write a partitioned dataset

The directory is a temporary one, cleaned up at the end of the section.

In [ ]:
if SPARK_AVAILABLE:
    import os
    import tempfile

    base_dir = tempfile.mkdtemp(prefix="de-interview-parquet-")
    parquet_path = os.path.join(base_dir, "orders_parquet")

    (
        orders
        .withColumn("order_date", F.to_date("order_ts"))
        .write
        .mode("overwrite")
        .partitionBy("order_date")
        .parquet(parquet_path)
    )
    print("wrote partitioned parquet to", parquet_path)
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### Read it back

**Expected output (with Spark):** the schema shown above, then `rows read back : 6`.

In [ ]:
if SPARK_AVAILABLE:
    read_back = spark.read.parquet(parquet_path)
    read_back.printSchema()
    print("rows read back        :", read_back.count())
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### Partition pruning in the plan

The `explain()` output should list only the matching partition directory in `PartitionFilters`.

**Expected output (with Spark):** `rows for 2026-10-01   : 2`.

In [ ]:
if SPARK_AVAILABLE:
    pruned = read_back.filter(F.col("order_date") == F.to_date(F.lit("2026-10-01")))
    print("rows for 2026-10-01   :", pruned.count())
    pruned.explain()  # the scan should list only the matching partition directory
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### The directories on disk

**Expected output (with Spark):** three `order_date=...` directories.

In [ ]:
if SPARK_AVAILABLE:
    import glob

    partition_dirs = sorted(
        os.path.basename(p) for p in glob.glob(os.path.join(parquet_path, "order_date=*"))
    )
    print("partition directories :", partition_dirs)
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### Expected output (with Spark)

```
self-check 10 passed: partitioned Parquet round-trip with pruning
```

In [ ]:
if SPARK_AVAILABLE:
    # --- self-check 10 --------------------------------------------------
    assert read_back.count() == 6
    assert pruned.count() == 2
    assert partition_dirs == [
        "order_date=2026-10-01",
        "order_date=2026-10-02",
        "order_date=2026-10-03",
    ]
    print("self-check 10 passed: partitioned Parquet round-trip with pruning")
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### Clean up the temporary directory

Nothing to assert here - it just keeps the workspace tidy.

In [ ]:
if SPARK_AVAILABLE:
    shutil.rmtree(base_dir, ignore_errors=True)
    print("removed", base_dir)
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


## 12. Caching / persist and when not to cache

### Concept

`cache()` = `persist(StorageLevel.MEMORY_AND_DISK)`. It stores the computed partitions so the
**next** action reuses them instead of recomputing the whole lineage.

`MEMORY_ONLY` drops partitions it cannot hold and recomputes them; `MEMORY_AND_DISK` spills to
disk; `DISK_ONLY` trades speed for memory. Always `unpersist()` when done - cached blocks
occupy executor memory that shuffle and joins also need.

### When NOT to cache

- the DataFrame is used once (cache is pure overhead);
- the source is already small/cheap, or you are going to `write` it immediately anyway;
- you cached "just in case" and the job now has memory pressure (the classic OOM culprit in long
  notebooks).

Also remember caching is **lazy** - the first action materialises it. Iterative algorithms (ML
training loops) and DataFrames reused 3+ times are the real wins.

### Expected output (with Spark)

```
is_cached after cache()      : True
rows (materialises cache)    : 3
rows from cache              : 3
is_cached after unpersist()  : False
persisted with MEMORY_AND_DISK: True
```

### Interview note

Contrast `cache` with `checkpoint`. Cache keeps the **lineage** and is lost when the session
dies; checkpoint truncates the lineage to a reliable store, which is what you need for very long
chains or exactly-once recovery.

### Cache an expensive aggregate

`is_cached` is `True` immediately after `cache()` (the plan is marked), but no data is stored
until the first action.

In [ ]:
if SPARK_AVAILABLE:
    from pyspark import StorageLevel

    expensive = (
        orders
        .withColumn("order_date", F.to_date("order_ts"))
        .groupBy("order_date")
        .agg(F.sum("amount").alias("revenue"))
        .cache()
    )
    print("is_cached after cache()      :", expensive.is_cached)
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### Materialise, then reuse

**Expected output (with Spark):**

```
rows (materialises cache)    : 3
rows from cache              : 3
```

In [ ]:
if SPARK_AVAILABLE:
    first = expensive.count()   # first action materialises the cached blocks
    second = expensive.count()  # served from memory/disk, no recompute
    print("rows (materialises cache)    :", first)
    print("rows from cache              :", second)
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### Unpersist and use an explicit storage level

**Expected output (with Spark):**

```
is_cached after unpersist()  : False
persisted with MEMORY_AND_DISK: True
```

In [ ]:
if SPARK_AVAILABLE:
    expensive.unpersist()
    print("is_cached after unpersist()  :", expensive.is_cached)

    persisted = orders.persist(StorageLevel.MEMORY_AND_DISK)
    persisted.count()
    print("persisted with MEMORY_AND_DISK:", persisted.is_cached)
    persisted.unpersist()
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### Expected output (with Spark)

```
self-check 11 passed: cache lifecycle correct
```

In [ ]:
if SPARK_AVAILABLE:
    # --- self-check 11 --------------------------------------------------
    assert first == 3 and second == 3
    assert not expensive.is_cached
    assert not persisted.is_cached
    print("self-check 11 passed: cache lifecycle correct")
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


## 13. Reading `explain()` plans

### Concept

`df.explain()` prints the physical plan; `explain(True)` also prints parsed, analysed and
optimised logical plans. Read it bottom-up (leaves to root).

### What to look for

- **Scan** - `FileScan parquet ... PushedFilters: [...]`; what was read and which predicates
  were pushed down. `PartitionFilters` shows partition pruning.
- **Filter / Project** - narrow work that stayed in-partition.
- **Exchange** - **a shuffle**. Any `Exchange` in the plan is your cost indicator.
- **HashAggregate** - a partial aggregate before the exchange, then a final one after.
- **BroadcastHashJoin** vs **SortMergeJoin** - whether the small side was broadcast.
- **AdaptiveSparkPlan** - AQE is enabled and may re-optimise at runtime.

### `EXPLAIN FORMATTED` as data

`spark.sql("EXPLAIN FORMATTED <query>")` returns the plan as a DataFrame, which is handy for
asserting on the plan in tests.

### Expected output (with Spark)

```
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[status#... ], functions=[sum(amount#...)])
   +- Exchange hashpartitioning(status#..., 8)
      +- HashAggregate(keys=[status#...], functions=[partial_sum(amount#...)])
         +- FileScan parquet ...
```

### Interview note

Being able to point at the `Exchange` and say "that is the shuffle, and here is why" is the
strongest practical signal in a Spark interview. Bring a real example: a `groupBy` on a
high-cardinality key with two `Exchange` nodes and a post-shuffle skew.

### Get the plan for a `GROUP BY`

**Expected output (with Spark):** the formatted plan, including an `Exchange`.

In [ ]:
if SPARK_AVAILABLE:
    orders.createOrReplaceTempView("orders")

    plan_rows = spark.sql(
        """
        EXPLAIN FORMATTED
        SELECT status, sum(amount) AS revenue
        FROM orders
        GROUP BY status
        """
    ).collect()
    plan_text = "\n".join(row[0] for row in plan_rows)
    print(plan_text)
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### Assert on the plan text

A `GROUP BY` is wide, so the plan must contain an `Exchange`. A test that asserts this is a
cheap regression guard for accidentally breaking predicate/pipeline behaviour.

**Expected output (with Spark):**

```
plan contains an Exchange (shuffle) as expected for GROUP BY
```

In [ ]:
if SPARK_AVAILABLE:
    lowered = plan_text.lower()
    assert len(plan_text) > 0
    assert "aggregate" in lowered
    # The GROUP BY is a wide transformation, so a shuffle must appear in the plan.
    assert "exchange" in lowered, plan_text
    print("plan contains an Exchange (shuffle) as expected for GROUP BY")
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


## 14. Adaptive Query Execution (AQE) and skew handling

### Concept

Since Spark 3.0, AQE re-optimises the plan **at runtime** using statistics from completed
shuffle stages.

### The three big features

1. **Coalesce post-shuffle partitions** - `spark.sql.adaptive.coalescePartitions.enabled=true`
   merges many small shuffle partitions into `advisoryPartitionSizeInBytes`-sized ones. This is
   the modern answer to "my job has 200 tiny tasks".
2. **Skew join handling** - `spark.sql.adaptive.skewJoin.enabled=true` detects
   disproportionately large partitions and splits them into sub-tasks (usually by salting
   internally), so one straggler task no longer defines the job duration.
3. **Convert sort-merge join to broadcast join** at runtime if a side turns out to be small.

### Manual skew handling

Without AQE, skew is handled manually: **salt** the hot keys (append a random suffix, explode
the other side), then re-aggregate.

### Expected output (with Spark)

```
AQE enabled               : true
coalescePartitions        : true
skewJoin enabled          : true
advisory partition size   : 64m
+---------+---+
|status   |n  |
+---------+---+
|paid     |3  |
|cancelled|1  |
|returned |1  |
|shipped  |1  |
+---------+---+
```

### Interview note

Say what AQE *cannot* fix - a single key holding most rows still ends up in one partition; AQE
splits it, but if the source read itself is skewed you may need to rebalance the input
(`repartition` by a salted key, or fix upstream). Also note AQE is disabled for Structured
Streaming in many versions, so streaming skew needs manual design.

### Turn the knobs

**Expected output (with Spark):** the four configuration values echoed back.

In [ ]:
if SPARK_AVAILABLE:
    spark.conf.set("spark.sql.adaptive.enabled", "true")
    spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
    spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
    spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "64m")

    print("AQE enabled               :", spark.conf.get("spark.sql.adaptive.enabled"))
    print("coalescePartitions        :", spark.conf.get("spark.sql.adaptive.coalescePartitions.enabled"))
    print("skewJoin enabled          :", spark.conf.get("spark.sql.adaptive.skewJoin.enabled"))
    print("advisory partition size   :", spark.conf.get("spark.sql.adaptive.advisoryPartitionSizeInBytes"))
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### Probe the skew

`explain()` shows an `AdaptiveSparkPlan`: the plan is re-planned after shuffle statistics
arrive.

In [ ]:
if SPARK_AVAILABLE:
    skew_probe = spark.sql(
        "SELECT status, count(*) AS n FROM orders GROUP BY status ORDER BY n DESC, status"
    )
    skew_probe.show()
    skew_probe.explain()
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### Expected output (with Spark)

```
self-check 12 passed: AQE flags set and skew probe returned every status
```

In [ ]:
if SPARK_AVAILABLE:
    # --- self-check 12 --------------------------------------------------
    assert spark.conf.get("spark.sql.adaptive.enabled") == "true"
    assert spark.conf.get("spark.sql.adaptive.skewJoin.enabled") == "true"
    assert skew_probe.count() == 4
    print("self-check 12 passed: AQE flags set and skew probe returned every status")
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


## 15. Small files and compaction

### Concept

Each Spark task writes one file per partition. A job with 2,000 partitions writes 2,000 files;
streaming or hourly ingestion multiplies that.

### Why small files hurt

- every file has footer/metadata overhead in the driver and in the scan plan;
- object stores (S3) list and open files with high per-request latency;
- query planning times out on millions of files.

### Fixes

`coalesce(n)` (narrow, no shuffle - it just merges partitions, best when reducing slightly) or
`repartition(n)` (full shuffle, even sizes - best when the current sizes are wildly uneven).
Then rewrite the table.

Dedicated compaction jobs (Databricks `OPTIMIZE`, Iceberg `rewrite_data_files`, Hudi
compaction) do the same thing periodically.

### Expected output (with Spark, illustrative)

```
small files before compaction: 24
files after compaction       : 2
rows preserved               : 6
```

The exact file counts depend on `repartition(8)` output and Spark's task count, but the point
is stable: fewer, larger files with identical rows.

### Interview note

Mention the **read** side too - `spark.sql.files.maxPartitionBytes` controls how much a single
task reads; too many small files means one task per file and a storm of tiny tasks. The usual
production answer is a scheduled compaction job, not ad-hoc `coalesce` in every consumer.

### Write deliberately fragmented files

`repartition(8)` inflates the task count so each day writes up to eight tiny files.

In [ ]:
if SPARK_AVAILABLE:
    small_base = tempfile.mkdtemp(prefix="de-interview-smallfiles-")
    raw_dir = os.path.join(small_base, "raw")
    compact_path = os.path.join(small_base, "compacted")

    for day in ("2026-10-01", "2026-10-02", "2026-10-03"):
        (
            orders
            .filter(F.to_date("order_ts") == F.to_date(F.lit(day)))
            .repartition(8)  # deliberately inflate the task -> file count
            .write
            .mode("overwrite")
            .parquet(os.path.join(raw_dir, f"ds={day}"))
        )
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### Count the files before compaction

**Expected output (with Spark, illustrative):** `small files before compaction: 24`.

In [ ]:
if SPARK_AVAILABLE:
    before = [
        os.path.join(root, f)
        for root, _, files in os.walk(raw_dir)
        for f in files
        if f.endswith(".parquet")
    ]
    print("small files before compaction:", len(before))
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### Compact with `coalesce`

The compacted copy goes to a sibling directory so re-reading `raw_dir` after the write cannot
pick up the compacted files (which would double-count the rows).

In [ ]:
if SPARK_AVAILABLE:
    raw = spark.read.option("recursiveFileLookup", "true").parquet(raw_dir)
    compacted = raw.coalesce(2)  # merge partitions without a shuffle
    compacted.write.mode("overwrite").parquet(compact_path)
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### Count the files after compaction

**Expected output (with Spark, illustrative):**

```
files after compaction       : 2
rows preserved               : 6
```

In [ ]:
if SPARK_AVAILABLE:
    after = [
        os.path.join(root, f)
        for root, _, files in os.walk(compact_path)
        for f in files
        if f.endswith(".parquet")
    ]
    print("files after compaction       :", len(after))
    print("rows preserved               :", compacted.count())
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


### Expected output (with Spark)

```
self-check 13 passed: compaction reduced file count and preserved rows
```

In [ ]:
if SPARK_AVAILABLE:
    # --- self-check 13 --------------------------------------------------
    assert compacted.count() == 6
    assert 0 < len(after) < len(before)
    print("self-check 13 passed: compaction reduced file count and preserved rows")
    shutil.rmtree(small_base, ignore_errors=True)
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


## 16. Structured Streaming sketch (code shown, never executed)

Structured Streaming treats a stream as an unbounded table you keep querying. Three things an
interviewer listens for:

- **`readStream` / `writeStream`** - the same DataFrame API as batch, so joins and aggregations
  transfer directly.
- **Checkpointing** - `checkpointLocation` persists offsets and state so a restart resumes where
  it stopped instead of reprocessing or losing data.
- **Output mode / watermark / trigger** - `append` vs `update` vs `complete`, and a watermark to
  bound state for event-time aggregations.

The block below is intentionally **not executed** by this notebook (it needs a Kafka broker and
a table sink). Read it as a reference shape.

```python
# Reference only - NOT executed by scripts/run_notebooks.py.
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType, StructType, TimestampType

order_schema = StructType() \
    .add("order_id", StringType()) \
    .add("customer_id", StringType()) \
    .add("amount", DoubleType()) \
    .add("status", StringType()) \
    .add("order_ts", TimestampType())

stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "orders")
    .option("startingOffsets", "latest")
    .load()
)
```

```python
parsed = (
    stream
    .select(F.from_json(F.col("value").cast("string"), order_schema).alias("o"))
    .select("o.*")
    .withWatermark("order_ts", "10 minutes")
)

query = (
    parsed
    .groupBy(F.window("order_ts", "5 minutes"), "status")
    .agg(F.count("*").alias("n"), F.sum("amount").alias("revenue"))
    .writeStream
    .outputMode("update")
    .option("checkpointLocation", "s3://company-checkpoints/orders_5m/")
    .trigger(processingTime="1 minute")
    .toTable("main.gold.orders_5m")  # or .foreachBatch(...) / .format("delta").start(path)
)

query.awaitTermination()
```

**Interview note:** the killer follow-up is **exactly-once**. Structured Streaming gives
end-to-end exactly-once only when the source is replayable (Kafka with committed offsets), the
checkpoint is durable, and the sink is idempotent/transactional (Delta, Iceberg, or a
`foreachBatch` merge keyed on a deterministic batch id). `update` mode without a watermark means
unbounded state.

## 17. Delta Lake and Apache Iceberg notes (mostly a Databricks conversation)

Both projects add a **transaction log / metadata layer** on top of Parquet, which is what turns
a directory of files into a reliable table. What they share:

- **ACID transactions and atomic metadata commits** - a failed write does not leave a
  half-visible table.
- **Time travel and snapshots** - query `VERSION AS OF` / `TIMESTAMP AS OF`.
- **Schema evolution and enforcement** - `mergeSchema`, `ALTER TABLE ... ADD COLUMN`.
- **`MERGE` / upsert** - the standard CDC pattern: match on the primary key, update matched
  rows, insert new ones, delete flagged ones.
- **Small-file management** - Delta `OPTIMIZE` (with `ZORDER BY` for data skipping), Iceberg
  `rewrite_data_files` + `rewrite_manifests`.
- **Statistics for data skipping** - per-file min/max, so `WHERE` on non-partition columns can
  still prune files.

### Differences worth calling out

| | Delta Lake | Apache Iceberg |
|---|---|---|
| Metadata | JSON + Parquet transaction log (`_delta_log`) | Metadata JSON + manifest files |
| Format spec | Delta protocol, originally Databricks | Open table spec, vendor-neutral |
| Hidden partitioning | Partition columns are real columns | `PARTITION BY` transforms (`days(ts)`, `bucket(id, 16)`) hide the column |
| Pushdown | Data skipping + ZORDER | Partition transforms + sort orders + manifests |
| Typical home | Databricks, Spark, some Trino/Flink | Trino, Flink, Spark, Snowflake, BigQuery |

### Databricks-specific habits

Unity Catalog for governance, `OPTIMIZE` + `ZORDER`, `VACUUM` to expire old files, `MERGE INTO`
for upserts, and `AUTO OPTIMIZE` / `AUTO COMPACT` for streaming sinks. On Databricks the
**Photon** engine and **predictive optimization** do much of the tuning, but partitioning, skew,
and file sizing are still design decisions you own.

**Expected output:** markdown only - nothing to execute.

**Interview note:** if asked "Delta or Iceberg?", answer with constraints: what query engines
must read the table, whether you need vendor neutrality, whether hidden partitioning and sort
orders matter, and what the team already runs. Then talk about the cost of migration - metadata
is the easy part; downstream `MERGE` jobs and permission models are not.

## Interview questions

Fifteen questions that cover the ground above. Answer each out loud before reading the answer.

**1. What is a shuffle, and why is it expensive?**

A shuffle redistributes rows across partitions so that rows sharing a key (or a partition
requirement) land together. It is expensive because it writes intermediate data to executor
local disk, transfers it across the network, reads it back, and can spill. It also forces a
stage boundary, so the whole upstream stage must complete first. Wide transformations cause
shuffles: `groupBy`, `join`, `distinct`, `repartition`, `orderBy`, and window functions with a
partition key. Narrow transformations (`filter`, `select`, `withColumn`, `union` of
same-partitioning data) do not.

**2. Narrow vs wide transformations - give examples and consequences.**

Narrow: each output partition depends on exactly one input partition (`map`, `filter`, `select`,
`withColumn`, `union`). No data movement; pipelined inside a stage. Wide: output partitions
depend on many input partitions (`groupBy`, `join`, `distinct`, `repartition`, `sort`). These
create a new stage, a shuffle, and a barrier. Rule of thumb: every `Exchange` node in
`explain()` marks a wide operation.

**3. When does a broadcast join help, and what are its limits?**

It helps when one side is small enough to fit in executor memory: Spark ships that side to every
executor and avoids shuffling the large side, turning a sort-merge join into a map-side broadcast
join. Limits: the broadcast side must fit in memory on **every** executor (default threshold
`spark.sql.autoBroadcastJoinThreshold=10MB`, and you can hint with `F.broadcast`); too large and
you get OOM or a silently ignored hint; broadcast also does not help if the *large* side is
skewed; and broadcasts are re-sent per stage/executor, so a "small" 500 MB table is not actually
small.

**4. What is Adaptive Query Execution and what problem does it solve?**

AQE re-optimises a query at runtime using actual shuffle statistics, instead of relying only on
plan-time estimates (which are often wrong). It coalesces many small post-shuffle partitions
(solving the 200-partition default problem), splits skewed partitions for skewed joins, and can
switch a sort-merge join to broadcast when a side turns out to be small. Enable with
`spark.sql.adaptive.enabled=true` (default since Spark 3.2) plus `coalescePartitions.enabled` and
`skewJoin.enabled`.

**5. How do you handle data skew, and what can AQE not fix?**

Detect it: one task takes far longer, one partition's spill/bytes dwarfs the others, `explain`
shows an unbalanced exchange. Fixes, in order: let AQE's skew join split the hot partitions;
salting the hot key (append a random suffix 0..N to the key on both sides, aggregate, then
re-aggregate without the suffix); broadcast the small side if skew comes from the join;
`repartition` on a better key; or fix the root cause (a null/placeholder key such as `"-1"`
holding most rows). AQE cannot fix a **skewed source read** - if one input file or one Kafka
partition holds most of the data, you must rebalance at ingestion.

**6. How do you choose partition columns, and what is over-partitioning?**

Partition on a low-to-medium cardinality column that is almost always in the `WHERE` clause
(date, region), that produces roughly 100 MB-1 GB per partition, and whose values are evenly
distributed. Over-partitioning is millions of tiny directories (e.g. partitioning by `user_id`
or by date+hour+minute), which inflates metadata, listing latency, and task count, and defeats
the point. Alternative for high-cardinality queries: leave the table unpartitioned/bucketed and
rely on Parquet/Delta/Iceberg statistics for file skipping.

**7. When should you cache, and when should you not?**

Cache when a DataFrame or an expensive intermediate is reused by 2-3+ actions, or in an
iterative algorithm. Do not cache when it is used once, when the source is already a small
in-memory table, when you are about to write the result anyway, or when it would push the job
into memory pressure (spilled cache is often slower than recomputing a cheap lineage). Caching
is lazy - the first action materialises it - and you should `unpersist()` when done.

**8. UDFs: why are they slow, and what are the alternatives?**

A Python UDF serialises the function and every row between the JVM and a Python worker, runs row
by row, prevents Catalyst from optimising the expression, and blocks whole-stage code
generation. Alternatives: built-in `pyspark.sql.functions` (always first choice); higher-order
SQL functions (`transform`, `filter`, `aggregate`) on arrays/maps; pandas UDFs (`@pandas_udf`)
which vectorise per batch via Arrow; or a Scala/Java UDF for JVM-native execution. Reserve plain
Python UDFs for cold paths.

**9. What is checkpointing, and how does it differ from caching?**

Checkpointing writes the DataFrame to a reliable store (HDFS/S3) and **truncates the lineage**,
so recovery does not replay an ever-growing chain of transformations. Caching keeps the lineage
and lives only in executor memory/disk for the session. Use checkpoints for very long lineage
chains, iterative algorithms, and streaming (where `checkpointLocation` stores offsets and
state). Checkpoint files must be cleaned up separately; they are not freed by `unpersist`.

**10. How does Structured Streaming achieve exactly-once?**

Exactly-once needs three things together: a **replayable source** with committed offsets (Kafka,
or a file source), a **durable checkpoint** recording the offsets and state for each micro-batch,
and an **idempotent/transactional sink** (Delta or Iceberg tables, or a `foreachBatch` upsert
keyed on the batch/version). The checkpoint lets a restart resume without reprocessing; the
transactional sink makes a partially written batch invisible or safely overwritten, so
at-least-once delivery becomes exactly-once state.

**11. Driver vs executor memory - what lives where, and what are the common OOMs?**

The driver runs the `SparkSession`, the DAG scheduler, the Catalyst planner, and collects
results. It owns memory for `collect()`, `toPandas()`, broadcast variables under the broadcast
threshold, and long-lived objects in notebook loops. Executors run tasks, hold shuffle data,
cached blocks, and broadcast joins. Common OOMs: `collect()`/`toPandas()` on a large result
(driver OOM); caching too much or broadcast joins that are not actually small (executor OOM);
`groupBy` on a skewed key with too few partitions (executor OOM/spill); and Python UDF workers
exhausting memory when each row builds large objects.

**12. How do you deal with small files?**

Prevent them at write time with sensible `repartition`/`coalesce` (one file per output
partition), and run scheduled compaction: Delta `OPTIMIZE`, Iceberg `rewrite_data_files`, Hudi
compaction, or a `coalesce`/`repartition` rewrite job. On the read side, bound
`spark.sql.files.maxPartitionBytes` and avoid one task per file. Small files hurt planning time,
object-store listing latency, and per-file metadata overhead, and they break predicate pushdown
efficiency because each file is opened individually.

**13. DataFrames vs RDDs - when would you still use an RDD?**

DataFrames go through Catalyst (rule-based + cost-based optimisation), Tungsten (code generation,
off-heap memory, compact binary layout), and a typed schema, so they are almost always faster and
safer. RDDs give you arbitrary Python/Scala objects and fine control over partitioning, which is
occasionally needed for custom iterative algorithms, untyped binary processing, or porting legacy
code. The interview answer: default to DataFrames, reach for RDDs (`df.rdd`) only when the API
genuinely cannot express the logic, and know that leaving the DataFrame API loses all optimizer
benefits.

**14. `repartition` vs `coalesce`?**

`repartition(n)` performs a **full shuffle** and produces `n` partitions of roughly even size;
use it to increase parallelism or to rebalance badly skewed data, optionally on a key
(`repartition(n, "key")`). `coalesce(n)` is narrow: it merges existing partitions without a
shuffle, so it can only decrease the count and the resulting partitions may be uneven; use it
before a write to reduce file count. Common pattern: `repartition` for balance, then
`coalesce`/`repartition` before writing to control file sizes.

**15. How do you debug a slow Spark job?**

Start from the Spark UI: find the longest stage, then the slowest task inside it, and compare its
input size and spill to its peers - a large gap means skew. Read `explain()` for `Exchange`
nodes, broadcast vs sort-merge joins, and full scans where you expected pruning. Check partition
counts (`spark.sql.shuffle.partitions`, input file counts), shuffle spill-to-disk metrics, GC
time, and whether a UDF or `collect()` is on the hot path. Then fix the highest-leverage thing:
broadcast/`repartition` the join, enable AQE, compact files, remove the UDF, or add a `filter`
earlier so less data is read.

## Running this notebook with Spark for real

```bash
# from the repo root
uv sync --extra notebooks --extra notebooks-spark
java -version                      # must be 17+
python3 scripts/run_notebooks.py notebooks/04_pyspark_concepts.ipynb
```

The runner executes every cell and saves the outputs back into the notebook. With PySpark and
Java present, `SPARK_AVAILABLE` becomes `True`, all SKIPPED messages turn into real results, and
the self-check assertion blocks run for real.

**Expected summary with Spark:** `Spark available: True`, `SparkSession stopped.`
**Expected summary here:** `Spark available: False`, all Spark cells reported SKIPPED.

In [ ]:
print("Spark available:", SPARK_AVAILABLE)
if SPARK_AVAILABLE:
    print("All Spark cells executed against a local SparkSession.")
    print("Re-run this notebook whenever you change the code to refresh the saved outputs.")
else:
    print("All Spark cells were skipped; every concept above still includes its expected output.")
    if not HAS_PYSPARK:
        print("Missing: pyspark  ->  uv sync --extra notebooks --extra notebooks-spark")
    if not (HAS_JAVA and JAVA_RUNS):
        print("Missing: a working Java 17+ runtime  ->  install Temurin or openjdk@17")


### Next steps

* `notebooks/03_sql_interview_questions.ipynb` - SQL window functions against a SQLite sandbox.
* `notebooks/05_interview_questions.ipynb` - a mixed question bank for mock interviews.

In [ ]:
if SPARK_AVAILABLE:
    spark.stop()
    print("SparkSession stopped.")
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")
